In [ ]:
#%pip install matplotlib seaborn scipy numpy pandas

# Presence Heatmap Generator## ObjectiveDevelop a **spatial density heatmap** by fusing two complementary data sources:1. **Time-stamped radar detections** — simulated target pings with (x, y) coordinates, timestamps, and detection confidence.2. **Simulated cell-sector data** — cell tower sectors with angular coverage, signal strength, and overlapping spatial footprints.### Pipeline1. **Simulate Radar Detections** — generate realistic detection events across a surveillance area2. **Simulate Cell-Sector Data** — model cell tower sectors with directional coverage3. **Data Fusion** — merge radar and cell-sector observations by time and location4. **Heatmap Generation** — produce spatial density maps highlighting movement concentration zones5. **Temporal Analysis** — animate or compare heatmaps across time windows

In [ ]:
# Import required libraries and set up environmenttry:    import warnings    warnings.filterwarnings('ignore')    import numpy as np    import pandas as pd    import matplotlib.pyplot as plt    import matplotlib.colors as mcolors    from matplotlib.patches import Wedge, Circle    from matplotlib.collections import PatchCollection    import seaborn as sns    from scipy.ndimage import gaussian_filter    from datetime import datetime, timedelta    # Reproducibility    SEED = 42    np.random.seed(SEED)    # Plotting defaults    plt.rcParams['figure.dpi'] = 120    sns.set_style('whitegrid')    print("All libraries imported successfully.")except ImportError as e:    print(f"Missing library – run the pip cell above first: {e}")    raise

In [ ]:
# ------------------------------------------------------------------# Global simulation parameters# ------------------------------------------------------------------# Surveillance area: 2 km × 2 km grid centred at (0, 0)AREA_SIZE    = 2000           # metresGRID_RES     = 20             # metres per grid cellGRID_BINS    = AREA_SIZE // GRID_RES   # 100 × 100 grid# Time window: 1 hour of activitySIM_START    = datetime(2026, 5, 16, 8, 0, 0)    # 08:00SIM_END      = datetime(2026, 5, 16, 9, 0, 0)    # 09:00SIM_DURATION = (SIM_END - SIM_START).total_seconds()# Number of radar detection eventsN_DETECTIONS = 5000# Number of cell towers and sectors per towerN_TOWERS     = 4SECTORS_PER  = 3     # typical tri-sector cell siteprint(f"Surveillance area : {AREA_SIZE}m × {AREA_SIZE}m")print(f"Grid resolution   : {GRID_RES}m  →  {GRID_BINS}×{GRID_BINS} cells")print(f"Time window       : {SIM_START} → {SIM_END}")print(f"Radar detections  : {N_DETECTIONS}")print(f"Cell towers       : {N_TOWERS}  ({SECTORS_PER} sectors each)")

In [ ]:
# ------------------------------------------------------------------# Step 1 — Simulate time-stamped radar detections# ------------------------------------------------------------------# Detections cluster around several "hotspots" (e.g. roads,# gathering areas) to create realistic spatial density variation.# Each detection carries: timestamp, x, y, confidence (0-1).# Define hotspots (centres of activity)hotspots = [    {'centre': (  400,  300), 'std': 120, 'weight': 0.30},   # market / plaza    {'centre': ( -300, -200), 'std':  80, 'weight': 0.20},   # checkpoint    {'centre': (  100, -500), 'std': 200, 'weight': 0.15},   # road corridor    {'centre': ( -600,  600), 'std': 150, 'weight': 0.15},   # residential area    {'centre': (    0,    0), 'std': 400, 'weight': 0.20},   # background / diffuse]detections = []half = AREA_SIZE / 2for hs in hotspots:    n = int(N_DETECTIONS * hs['weight'])    cx, cy = hs['centre']    xs = np.random.normal(cx, hs['std'], n)    ys = np.random.normal(cy, hs['std'], n)    # Clip to surveillance area    xs = np.clip(xs, -half, half)    ys = np.clip(ys, -half, half)    # Random timestamps within the simulation window    ts = SIM_START + np.array([timedelta(seconds=s)                               for s in np.random.uniform(0, SIM_DURATION, n)])    # Detection confidence — higher near hotspot centre    dist = np.sqrt((xs - cx)**2 + (ys - cy)**2)    conf = np.clip(1.0 - dist / (3 * hs['std']) + np.random.normal(0, 0.08, n),                   0.2, 1.0)    for i in range(n):        detections.append({            'timestamp': ts[i],            'x': xs[i],            'y': ys[i],            'confidence': conf[i],        })df_radar = pd.DataFrame(detections).sort_values('timestamp').reset_index(drop=True)print(f"Radar detections generated: {len(df_radar)}")print(df_radar.head(10))print(f"\nConfidence stats:\n{df_radar['confidence'].describe()}")

In [ ]:
# Scatter plot of raw radar detections coloured by confidencefig, ax = plt.subplots(figsize=(8, 8))sc = ax.scatter(df_radar['x'], df_radar['y'],                c=df_radar['confidence'], cmap='YlOrRd',                s=4, alpha=0.5, edgecolors='none')plt.colorbar(sc, ax=ax, label='Detection Confidence')ax.set_xlim(-AREA_SIZE/2, AREA_SIZE/2)ax.set_ylim(-AREA_SIZE/2, AREA_SIZE/2)ax.set_xlabel('X (metres)')ax.set_ylabel('Y (metres)')ax.set_title('Raw Radar Detections', fontsize=14, fontweight='bold')ax.set_aspect('equal')plt.tight_layout()plt.show()

In [ ]:
# ------------------------------------------------------------------# Step 2 — Simulate cell-sector data# ------------------------------------------------------------------# Each tower has N sectors defined by: tower position, azimuth angle,# beam width, and max range.  For every radar detection that falls# inside a sector, we assign a "cell signal strength" that decays# with distance from the tower.tower_positions = [    ( 500,  500),    (-500,  500),    (-500, -500),    ( 500, -500),]sectors = []for t_idx, (tx, ty) in enumerate(tower_positions):    for s_idx in range(SECTORS_PER):        azimuth   = s_idx * (360 / SECTORS_PER)   # degrees        beamwidth = 360 / SECTORS_PER              # degrees        max_range = np.random.uniform(700, 1100)   # metres        sectors.append({            'tower_id': t_idx,            'sector_id': s_idx,            'tx': tx, 'ty': ty,            'azimuth': azimuth,            'beamwidth': beamwidth,            'max_range': max_range,        })df_sectors = pd.DataFrame(sectors)print(f"Cell sectors defined: {len(df_sectors)}")print(df_sectors)

In [ ]:
# Visualise tower positions and sector footprintsfig, ax = plt.subplots(figsize=(8, 8))cmap_sectors = plt.cm.Set3for idx, row in df_sectors.iterrows():    theta1 = row['azimuth'] - row['beamwidth'] / 2    theta2 = row['azimuth'] + row['beamwidth'] / 2    wedge = Wedge((row['tx'], row['ty']), row['max_range'],                  theta1, theta2, alpha=0.15,                  color=cmap_sectors(row['tower_id'] / N_TOWERS))    ax.add_patch(wedge)# Tower markersfor t_idx, (tx, ty) in enumerate(tower_positions):    ax.plot(tx, ty, 's', markersize=10, color='black', zorder=5)    ax.annotate(f'T{t_idx}', (tx, ty), textcoords='offset points',                xytext=(8, 8), fontsize=10, fontweight='bold')ax.set_xlim(-AREA_SIZE/2 - 200, AREA_SIZE/2 + 200)ax.set_ylim(-AREA_SIZE/2 - 200, AREA_SIZE/2 + 200)ax.set_xlabel('X (metres)')ax.set_ylabel('Y (metres)')ax.set_title('Cell Tower Sectors', fontsize=14, fontweight='bold')ax.set_aspect('equal')plt.tight_layout()plt.show()

In [ ]:
# ------------------------------------------------------------------# Step 3 — Fuse radar detections with cell-sector data# ------------------------------------------------------------------# For each radar detection, determine which cell sector(s) it falls# inside and compute a signal-strength weight.  The fused "density# score" for each detection = radar confidence × cell signal strength.def point_in_sector(px, py, tx, ty, azimuth, beamwidth, max_range):    """Check if point (px, py) falls inside a sector wedge."""    dx = px - tx    dy = py - ty    dist = np.sqrt(dx**2 + dy**2)    if dist > max_range:        return False, 0.0    angle = (np.degrees(np.arctan2(dy, dx)) + 360) % 360    half_bw = beamwidth / 2    a_start = (azimuth - half_bw + 360) % 360    a_end   = (azimuth + half_bw + 360) % 360    if a_start < a_end:        inside = a_start <= angle <= a_end    else:  # wraps around 0°        inside = angle >= a_start or angle <= a_end    if inside:        # Signal strength: inverse-square decay, normalised 0-1        strength = max(0.0, 1.0 - (dist / max_range)**2)        return True, strength    return False, 0.0# Compute fused scoresfused_scores = []sector_hits  = []for _, det in df_radar.iterrows():    px, py = det['x'], det['y']    best_strength = 0.0    hit_sector = -1    for _, sec in df_sectors.iterrows():        inside, strength = point_in_sector(            px, py, sec['tx'], sec['ty'],            sec['azimuth'], sec['beamwidth'], sec['max_range'])        if inside and strength > best_strength:            best_strength = strength            hit_sector = int(sec['tower_id'] * SECTORS_PER + sec['sector_id'])    # Fused density score = radar confidence × cell signal strength    # If outside all sectors, use radar confidence only (strength = 0.3 baseline)    cell_weight = best_strength if best_strength > 0 else 0.3    fused_score = det['confidence'] * cell_weight    fused_scores.append(fused_score)    sector_hits.append(hit_sector)df_radar['cell_strength']  = [fs / df_radar['confidence'].iloc[i]                               if df_radar['confidence'].iloc[i] > 0 else 0.3                               for i, fs in enumerate(fused_scores)]df_radar['fused_score']    = fused_scoresdf_radar['sector_hit']     = sector_hitspct_in_sector = 100 * (df_radar['sector_hit'] >= 0).sum() / len(df_radar)print(f"Detections inside a cell sector: {pct_in_sector:.1f}%")print(f"\nFused score stats:\n{df_radar['fused_score'].describe()}")print(df_radar.head(10))

In [ ]:
# ------------------------------------------------------------------# Step 4 — Build spatial density heatmap# ------------------------------------------------------------------# Accumulate fused scores into a 2-D grid, then smooth with a# Gaussian kernel for visual clarity.def build_heatmap(df, score_col='fused_score', grid_bins=GRID_BINS,                  area_size=AREA_SIZE, sigma=2.0):    """Bin detections into a 2-D grid and apply Gaussian smoothing."""    half = area_size / 2    heatmap = np.zeros((grid_bins, grid_bins), dtype=np.float64)    for _, row in df.iterrows():        xi = int((row['x'] + half) / area_size * grid_bins)        yi = int((row['y'] + half) / area_size * grid_bins)        xi = np.clip(xi, 0, grid_bins - 1)        yi = np.clip(yi, 0, grid_bins - 1)        heatmap[yi, xi] += row[score_col]    # Gaussian smoothing    heatmap = gaussian_filter(heatmap, sigma=sigma)    return heatmapheatmap_fused = build_heatmap(df_radar, score_col='fused_score', sigma=2.5)print(f"Heatmap shape  : {heatmap_fused.shape}")print(f"Max density    : {heatmap_fused.max():.2f}")print(f"Mean density   : {heatmap_fused.mean():.2f}")

In [ ]:
# Render the fused presence heatmapfig, ax = plt.subplots(figsize=(10, 8))half = AREA_SIZE / 2extent = [-half, half, -half, half]im = ax.imshow(heatmap_fused, origin='lower', extent=extent,               cmap='inferno', interpolation='bilinear', aspect='equal')cbar = plt.colorbar(im, ax=ax, shrink=0.8, pad=0.02)cbar.set_label('Fused Density Score', fontsize=12)# Overlay tower positionsfor t_idx, (tx, ty) in enumerate(tower_positions):    ax.plot(tx, ty, 's', markersize=8, color='cyan', zorder=5,            markeredgecolor='white', markeredgewidth=1.5)    ax.annotate(f'T{t_idx}', (tx, ty), textcoords='offset points',                xytext=(8, 8), fontsize=9, fontweight='bold', color='white')# Overlay hotspot centresfor hs in hotspots:    cx, cy = hs['centre']    ax.plot(cx, cy, 'x', markersize=10, color='lime', markeredgewidth=2, zorder=5)ax.set_xlabel('X (metres)', fontsize=12)ax.set_ylabel('Y (metres)', fontsize=12)ax.set_title('Presence Heatmap — Fused Radar + Cell-Sector Data',             fontsize=14, fontweight='bold')plt.tight_layout()plt.show()

In [ ]:
# Side-by-side: radar-only density vs fused densityheatmap_radar_only = build_heatmap(df_radar, score_col='confidence', sigma=2.5)fig, axes = plt.subplots(1, 2, figsize=(16, 7))half = AREA_SIZE / 2extent = [-half, half, -half, half]for ax, hmap, title in zip(axes,        [heatmap_radar_only, heatmap_fused],        ['Radar-Only Density (confidence)', 'Fused Density (radar × cell)']):    im = ax.imshow(hmap, origin='lower', extent=extent,                   cmap='inferno', interpolation='bilinear', aspect='equal')    plt.colorbar(im, ax=ax, shrink=0.75, pad=0.02)    ax.set_title(title, fontsize=13, fontweight='bold')    ax.set_xlabel('X (metres)')    ax.set_ylabel('Y (metres)')    # towers    for t_idx, (tx, ty) in enumerate(tower_positions):        ax.plot(tx, ty, 's', markersize=7, color='cyan',                markeredgecolor='white', markeredgewidth=1)plt.suptitle('Radar-Only vs Fused Heatmap Comparison', fontsize=15,             fontweight='bold', y=1.02)plt.tight_layout()plt.show()

In [ ]:
# ------------------------------------------------------------------# Step 5 — Temporal analysis: heatmaps by 15-minute windows# ------------------------------------------------------------------# Split the 1-hour simulation into four 15-minute windows and plot# separate heatmaps to show how movement patterns evolve over time.time_windows = [    (SIM_START + timedelta(minutes=0),  SIM_START + timedelta(minutes=15)),    (SIM_START + timedelta(minutes=15), SIM_START + timedelta(minutes=30)),    (SIM_START + timedelta(minutes=30), SIM_START + timedelta(minutes=45)),    (SIM_START + timedelta(minutes=45), SIM_START + timedelta(minutes=60)),]fig, axes = plt.subplots(1, 4, figsize=(20, 5))half = AREA_SIZE / 2extent = [-half, half, -half, half]# Compute global max across all windows for consistent colour scalehmaps = []for t_start, t_end in time_windows:    mask = (df_radar['timestamp'] >= t_start) & (df_radar['timestamp'] < t_end)    hm = build_heatmap(df_radar[mask], score_col='fused_score', sigma=2.5)    hmaps.append(hm)vmax = max(h.max() for h in hmaps)for ax, hm, (t_start, t_end) in zip(axes, hmaps, time_windows):    im = ax.imshow(hm, origin='lower', extent=extent,                   cmap='inferno', interpolation='bilinear',                   aspect='equal', vmin=0, vmax=vmax)    ax.set_title(f"{t_start.strftime('%H:%M')}–{t_end.strftime('%H:%M')}",                 fontsize=12, fontweight='bold')    ax.set_xlabel('X (m)')    if ax == axes[0]:        ax.set_ylabel('Y (m)')    else:        ax.set_yticks([])fig.suptitle('Temporal Presence Heatmaps (15-min windows)',             fontsize=15, fontweight='bold', y=1.04)fig.colorbar(im, ax=axes, shrink=0.8, pad=0.02, label='Fused Density Score')plt.tight_layout()plt.show()

In [ ]:
# ------------------------------------------------------------------# Step 6 — Identify top concentration zones from the heatmap# ------------------------------------------------------------------# Find the grid cells with the highest fused density and report# their coordinates.flat_idx = np.argsort(heatmap_fused.ravel())[::-1]top_n = 10half = AREA_SIZE / 2print("Top Concentration Zones (highest fused density):")print("-" * 55)print(f"{'Rank':>4}  {'X (m)':>8}  {'Y (m)':>8}  {'Density':>10}")print("-" * 55)for rank, fi in enumerate(flat_idx[:top_n], 1):    yi, xi = divmod(fi, GRID_BINS)    x_m = xi * GRID_RES - half + GRID_RES / 2    y_m = yi * GRID_RES - half + GRID_RES / 2    print(f"{rank:4d}  {x_m:8.1f}  {y_m:8.1f}  {heatmap_fused.ravel()[fi]:10.2f}")print(f"\nTotal detections            : {len(df_radar)}")print(f"Detections in cell sectors  : {int((df_radar['sector_hit'] >= 0).sum())}")print(f"Mean fused score            : {df_radar['fused_score'].mean():.4f}")print(f"Max  fused score            : {df_radar['fused_score'].max():.4f}")

# Project Summary and Conclusions## Heatmap Generation- Fused **5,000 simulated radar detections** with **4 tri-sector cell towers** (12 total sectors)- Spatial density grid: 100 × 100 cells covering a 2 km × 2 km area- Gaussian smoothing (σ = 2.5 cells) for visually coherent output## Data Fusion Approach- Each detection is scored: `fused_score = radar_confidence × cell_signal_strength`- Signal strength decays with inverse-square distance from the serving tower- Detections outside all cell sectors receive a baseline weight of 0.3## Key Findings- The fused heatmap sharpens hotspot contrast compared to radar-only density- Cell-sector weighting suppresses low-confidence detections far from towers- Temporal slicing reveals how movement concentration shifts over time## Recommendations1. **Real data integration**:   - Replace synthetic hotspots with actual radar track data   - Ingest real CDR (Call Detail Record) or cell-sector logs2. **Advanced fusion**:   - Bayesian or Kalman-filter fusion for multi-source weighting   - Incorporate Notebook 2 classifications (human / vehicle / drone / animal) as a class-specific heatmap layer3. **Deployment**:   - Stream radar detections in real time → rolling heatmap updates   - Export density maps as GeoTIFF for GIS overlay## Next Steps- [ ] Integrate with Notebook 1 (signal classifier) and Notebook 2 (micro-Doppler) for end-to-end pipeline- [ ] Add class-specific heatmap layers (e.g. humans only, vehicles only)- [ ] Implement real-time streaming mode with sliding time windows- [ ] Export to geospatial formats (GeoJSON, GeoTIFF)